## Bijdrage Raoul Buffels — DeepLearningTeam10

Deze Citi Bike-uitwerking is aangeleverd als de bijdrage van **Raoul Buffels**. Teamleden: Jorre Van Dyck, Milan Wouters, Raoul Buffels en Andrew Noeyens. De concrete menselijke taakverdeling moet het team zelf aanvullen. **AI-gebruik:** Codex hielp met implementatie, uitleg, tests, uitvoering en verpakking.

De codecellen zijn uitgevoerd in de oorspronkelijke lokale `Citybike_full`-omgeving op 9 oktober 2026. Deze export behoudt die echte uitvoer; alleen deze Markdown-cel is toegevoegd. Lokale paden, timings en fingerprints in de rapporten beschrijven de oorspronkelijke uitvoering. Start voor heruitvoering vanuit `SolutionRaoul/Citybike` of de map `notebooks` met de bijbehorende Python 3.11-omgeving. AWS is een afzonderlijke latere taak.

# LightGBM

Bijdragen: door team in te vullen. De training gebeurt offline via `run.py train`. Dit notebook bewaart echte scores en configuraties; geen eindtest wordt voor tuning gebruikt.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
locations = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'Citybike_full']
ROOT = next((p for p in locations if (p / 'citibike').exists()), None)
if ROOT is None: raise RuntimeError('Start dit notebook vanuit Citybike_full of notebooks.')
sys.path.insert(0, str(ROOT))
from citibike.config import REPORTS, DATA, MODELS
from citibike.datasets import manifest
from citibike.plots import show
def report(name):
    return json.loads((REPORTS / name).read_text(encoding='utf-8'))
def table(name):
    display(pd.read_csv(REPORTS / name, dtype={'start_station_id':'string','end_station_id':'string','station_id':'string'}))


De kern van iedere trainingsstap is hieronder weergegeven. De CLI herhaalt dit voor dezelfde opgeslagen folds, fit preprocessing alleen op de trainingsrijen en bewaart de werkelijke scores. Dit rapportnotebook voert na de eindtest geen nieuwe selectie uit.

```python
fitted = clone(candidate).fit(training[FEATURES], training.rides)
predicted = fitted.predict(validation[FEATURES])
scores = metrics(validation.rides, predicted)
```

In [2]:
from citibike.modeling import candidates
from citibike.features import FEATURES
import joblib
for name in ['lightgbm', 'tuned_lightgbm']:
    if (MODELS/(name+'.joblib')).exists():
        model=joblib.load(MODELS/(name+'.joblib'))
        display(Markdown('**Opgeslagen estimator: '+name+'**'))
        display(Markdown(str(model.estimator_)))

**Opgeslagen estimator: lightgbm**

LGBMRegressor(deterministic=True, force_col_wise=True, learning_rate=0.05,
              min_child_samples=40, n_estimators=250, n_jobs=2, random_state=42,
              verbosity=-1)

**Opgeslagen estimator: tuned_lightgbm**

LGBMRegressor(deterministic=True, force_col_wise=True, min_child_samples=100,
              n_estimators=200, n_jobs=2, random_state=42, verbosity=-1)

Dezelfde chronologische validatie en operationele voorspelfunctie maken de vergelijking eerlijk.

In [3]:
comparison=pd.read_csv(REPORTS/'model_comparison.csv')
display(comparison[comparison.name.isin(['lightgbm', 'tuned_lightgbm'])])

,name,RMSE,MAE,R2,train_RMSE,train_MAE,train_R2,cv_RMSE,fit_cv_seconds
3,lightgbm,1952.244762,1152.417360,0.841616,778.042812,432.043207,0.928749,1423.037130,10.576864
4,tuned_lightgbm,1972.512094,1204.596739,0.838310,744.497812,413.174794,0.934760,1383.390475,8.319329


Een lagere trainingsfout kan overfit betekenen. Kies op validatie en bekijk de CV-spreiding en looptijd.

In [4]:
records=report('model_comparison.json')
for record in records:
    if record['name'] in ['lightgbm', 'tuned_lightgbm']:
        display(Markdown('**'+record['name']+'**'))
        display(pd.DataFrame(record['cv']))
        display(Markdown(str(record['parameters'])))

**lightgbm**

,fold,RMSE,MAE,R2,seconds
0,1,1097.776580,705.432505,0.737440,1.652942
1,2,1132.317392,754.439793,0.737079,1.518061
2,3,1064.697349,720.954818,0.627176,1.488229
3,4,1840.632710,1185.945932,0.720506,1.655749
4,5,1979.761620,1224.885053,0.786534,1.730756


{'columns': None, 'estimator__boosting_type': 'gbdt', 'estimator__class_weight': None, 'estimator__colsample_bytree': 1.0, 'estimator__importance_type': 'split', 'estimator__learning_rate': 0.05, 'estimator__max_depth': -1, 'estimator__min_child_samples': 40, 'estimator__min_child_weight': 0.001, 'estimator__min_split_gain': 0.0, 'estimator__n_estimators': 250, 'estimator__n_jobs': 2, 'estimator__num_leaves': 31, 'estimator__objective': None, 'estimator__random_state': 42, 'estimator__reg_alpha': 0.0, 'estimator__reg_lambda': 0.0, 'estimator__subsample': 1.0, 'estimator__subsample_for_bin': 200000, 'estimator__subsample_freq': 0, 'estimator__verbosity': -1, 'estimator__deterministic': True, 'estimator__force_col_wise': True, 'estimator': 'LGBMRegressor(deterministic=True, force_col_wise=True, learning_rate=0.05,\n              min_child_samples=40, n_estimators=250, n_jobs=2, random_state=42,\n              verbosity=-1)'}

**tuned_lightgbm**

,fold,RMSE,MAE,R2,seconds
0,1,1068.037000,694.414172,0.751474,1.239121
1,2,1108.517185,736.371192,0.748015,1.177321
2,3,1068.703075,725.227128,0.624365,1.324060
3,4,1784.384115,1141.539329,0.737327,1.231037
4,5,1887.311002,1153.186007,0.806005,1.295385


{'columns': None, 'estimator__boosting_type': 'gbdt', 'estimator__class_weight': None, 'estimator__colsample_bytree': 1.0, 'estimator__importance_type': 'split', 'estimator__learning_rate': 0.1, 'estimator__max_depth': -1, 'estimator__min_child_samples': 100, 'estimator__min_child_weight': 0.001, 'estimator__min_split_gain': 0.0, 'estimator__n_estimators': 200, 'estimator__n_jobs': 2, 'estimator__num_leaves': 31, 'estimator__objective': None, 'estimator__random_state': 42, 'estimator__reg_alpha': 0.0, 'estimator__reg_lambda': 0.0, 'estimator__subsample': 1.0, 'estimator__subsample_for_bin': 200000, 'estimator__subsample_freq': 0, 'estimator__verbosity': -1, 'estimator__deterministic': True, 'estimator__force_col_wise': True, 'estimator': 'LGBMRegressor(deterministic=True, force_col_wise=True, min_child_samples=100,\n              n_estimators=200, n_jobs=2, random_state=42, verbosity=-1)'}

In [5]:
for path in sorted(REPORTS.glob('*_tuning.csv')):
    if path.stem.split('_tuning')[0] in ['lightgbm']:
        display(pd.read_csv(path)[['configuration','cv_RMSE','parameters']])

,configuration,cv_RMSE,parameters
0,1,1423.633347,"{""estimator__num_leaves"": 15, ""estimator__n_es..."
1,2,1503.104722,"{""estimator__num_leaves"": 15, ""estimator__n_es..."
2,3,1410.961752,"{""estimator__num_leaves"": 31, ""estimator__n_es..."
3,4,1421.446054,"{""estimator__num_leaves"": 31, ""estimator__n_es..."
4,5,1499.484772,"{""estimator__num_leaves"": 15, ""estimator__n_es..."
5,6,1438.275587,"{""estimator__num_leaves"": 31, ""estimator__n_es..."
6,7,1461.382875,"{""estimator__num_leaves"": 31, ""estimator__n_es..."
7,8,1410.433592,"{""estimator__num_leaves"": 31, ""estimator__n_es..."
8,9,1437.919426,"{""estimator__num_leaves"": 31, ""estimator__n_es..."
9,10,1455.752125,"{""estimator__num_leaves"": 15, ""estimator__n_es..."
